In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
pd.set_option('future.no_silent_downcasting', True)

def processar_dados_estoque_vendas(url_sales, url_inventory, data_inicial_str, caminho_saida_csv):
    """
    Processa dados de estoque e vendas, retornando um DataFrame final consolidado e salvando-o como CSV.
    
    Args:
        url_sales (str): Link compartilhável da planilha de vendas (Google Sheets).
        url_inventory (str): Link compartilhável da planilha de estoque (Google Sheets).
        data_inicial_str (str): Data inicial para análise no formato 'AAAA-MM-DD'.
        caminho_saida_csv (str): Caminho completo para salvar o arquivo CSV resultante.

    Returns:
        pd.DataFrame: DataFrame consolidado com os dados processados.
    """
    # Carrega os dados das planilhas
    df_vendas = pd.read_csv(url_sales)
    df_estoque = pd.read_csv(url_inventory)

    # Filtra os dados relevantes
    df_vendas = df_vendas.loc[df_vendas['Local'].isin(['Nacional', 'Internacional'])]
    df_vendas['Data'] = pd.to_datetime(df_vendas['Data'], dayfirst=True)

    # Define datas de análise
    data_inicial = pd.to_datetime(data_inicial_str)
    data_final = datetime.now() - timedelta(days=1)

    df_vendas = df_vendas[(df_vendas['Data'] >= data_inicial) & (df_vendas['Data'] <= data_final)]

    dias_no_periodo = int((data_final - data_inicial).days)

    # Adiciona colunas de mês e agrupa os dados
    df_vendas['Mês'] = df_vendas['Data'].dt.strftime('%B')
    df_saidas_mensais = df_vendas.groupby(['SKU', 'Local', 'Mês'])['Quantidade'].sum().reset_index()

    # Pivot para organizar os dados de saída
    df_saidas_pivot = df_saidas_mensais.pivot(index='SKU', columns=['Local', 'Mês'], values='Quantidade')
    df_saidas_pivot.columns = [
        f"Saídas {estoque} - {mes}" if isinstance(mes, str) else coluna
        for coluna, (estoque, mes) in zip(df_saidas_pivot.columns, df_saidas_pivot.columns)
    ]

    # Fusão dos DataFrames de vendas e estoque
    df_resultado = pd.merge(df_estoque, df_saidas_pivot, on='SKU', how='left').fillna(0)

    # Soma de saídas e cálculo de médias diárias
    df_resultado['Total Saídas Internacional'] = df_resultado.filter(like='Saídas Internacional').sum(axis=1)
    df_resultado['Total Saídas Nacional'] = df_resultado.filter(like='Saídas Nacional').sum(axis=1)
    df_resultado['Média Diária de Vendas Nacional'] = (df_resultado['Total Saídas Nacional'] / dias_no_periodo).round(2)
    df_resultado['Média Diária de Vendas Internacional'] = (df_resultado['Total Saídas Internacional'] / dias_no_periodo).round(2)

    # Cálculo da cobertura de estoque
    df_resultado['Cobertura de Estoque Nacional'] = np.where(
        df_resultado['Média Diária de Vendas Nacional'] > 0,
        df_resultado['Nacional - Quantidade'] / df_resultado['Média Diária de Vendas Nacional'],
        None
    )
    df_resultado['Cobertura de Estoque Internacional'] = np.where(
        df_resultado['Média Diária de Vendas Internacional'] > 0,
        df_resultado['Internacional - Quantidade'] / df_resultado['Média Diária de Vendas Internacional'],
        None
    )

    # Preenchimento de NaNs com valor máximo e conversão para inteiros
    df_resultado['Cobertura de Estoque Nacional'] = df_resultado['Cobertura de Estoque Nacional'].fillna(
        df_resultado['Cobertura de Estoque Nacional'].max()).astype(int).round(1)
    df_resultado['Cobertura de Estoque Internacional'] = df_resultado['Cobertura de Estoque Internacional'].fillna(
        df_resultado['Cobertura de Estoque Internacional'].max()).astype(int).round(1)

    # Conversão de colunas específicas para numéricas
    colunas_para_converter = [col for col in df_resultado.columns if col.startswith(('Saídas', 'Total', 'Média', 'Cobertura'))]
    df_resultado[colunas_para_converter] = df_resultado[colunas_para_converter].apply(pd.to_numeric, errors='coerce')

    # Salvar o resultado no arquivo CSV
    df_resultado.to_csv(caminho_saida_csv, index=False, sep=";", decimal=",")

    return df_resultado

# Exemplo de uso:
df_final = processar_dados_estoque_vendas(
    url_sales="https://docs.google.com/spreadsheets/d/1u7TNCUlMPfIB2SsbjIbmJEnwpVuHo6FD_vQ-vvpCIIc/export?format=csv&gid=1663278988",
    url_inventory="https://docs.google.com/spreadsheets/d/1u7TNCUlMPfIB2SsbjIbmJEnwpVuHo6FD_vQ-vvpCIIc/export?format=csv&gid=1547581836",
    data_inicial_str="2024-11-01",
    caminho_saida_csv=r"C:\Users\GabrielCielo\Desktop\consolatio\Databases\testefuncao.csv"
)
